In [442]:
# Load the extension that allows us to compile CUDA code in python notebooks
# Documentation is here: https://nvcc4jupyter.readthedocs.io/en/latest/
# !pip install git+https://github.com/andreinechaev/nvcc4jupyter.git
%load_ext nvcc4jupyter

The nvcc4jupyter extension is already loaded. To reload it, use:
  %reload_ext nvcc4jupyter


In [443]:
%%cuda_group_save -g "source" -n "data_types.h"
/**
 * A collection of commonly used data types throughout this project.
 */
#pragma once

#include <stdint.h> // uint32_t

using element_t = uint32_t;

In [444]:
%%cuda_group_save -g "source" -n "cuda_common.h"
/**
 * Standard macros that can be useful for error checking.
 * https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__ERROR.html
 */
#pragma once

#include <cuda.h>

#define CUDA_CALL(exp)                                       \
    do {                                                     \
        cudaError res = (exp);                               \
        if(res != cudaSuccess) {                             \
            printf("Error at %s:%d\n %s\n",                  \
                __FILE__,__LINE__, cudaGetErrorString(res)); \
           exit(EXIT_FAILURE);                               \
        }                                                    \
    } while(0)

#define CHECK_ERROR(msg)                                             \
    do {                                                             \
        cudaError_t err = cudaGetLastError();                        \
        if(cudaSuccess != err) {                                     \
            printf("Error (%s) at %s:%d\n %s\n",                     \
                (msg), __FILE__, __LINE__, cudaGetErrorString(err)); \
            exit(EXIT_FAILURE);                                      \
        }                                                            \
    } while (0)

In [445]:
%%cuda_group_save -g "source" -n "data_generator.h"
/**
 * Functions for generating random input data with a fixed seed
 */
#pragma once

#include <random>  // for std::mt19937, std::uniform_int_distribution
#include <vector>

#include "data_types.h"

namespace csc485b {
namespace a1 {

/**
 * Generates and returns a vector of random uniform data of a given length, n,
 * for any integral type. Input range will be [0, 2n].
 */
template < typename T >
std::vector< T > generate_uniform( std::size_t n )
{
    // for details of random number generation, see:
    // https://en.cppreference.com/w/cpp/numeric/random/uniform_int_distribution
    std::size_t random_seed = 20240916;  // use magic seed
    std::mt19937 rng( random_seed );     // use mersenne twister generator
    std::uniform_int_distribution<> distrib(0, 2 * n);

    std::vector< T > random_data( n ); // init array
    std::generate( std::begin( random_data )
                 , std::end  ( random_data )
                 , [ &rng, &distrib ](){ return static_cast< T >( distrib( rng ) ); });

    return random_data;
}


} // namespace a1
} // namespace csc485b

In [446]:
%%cuda_group_save -g "source" -n "algorithm_choices.h"
#pragma once

#include <vector>

#include "data_types.h"

namespace csc485b {
namespace a1 {
namespace cpu {

void run_cpu_baseline( std::vector< element_t > data, std::size_t switch_at, std::size_t n );

} // namespace cpu


namespace gpu {

void run_gpu_soln( std::vector< element_t > data, std::size_t switch_at, std::size_t n );

} // namespace gpu
} // namespace a1
} // namespace csc485b

In [447]:
%%cuda_group_save -g "source" -n "cpu_baseline.cu"
/**
 * CPU methods that the GPU should outperform.
 */

#include "algorithm_choices.h"

#include <algorithm> // std::sort()
#include <chrono>    // for timing
#include <iostream>  // std::cout, std::endl

namespace csc485b {
namespace a1      {
namespace cpu     {

/**
 * Simple solution that just sorts the whole array with a built-in sort
 * function and then resorts the last portion in the opposing order with
 * a second call to that same built-in sort function.
 */
void opposing_sort( element_t * data, std::size_t invert_at_pos, std::size_t num_elements )
{
    std::sort( data, data + num_elements, std::less< element_t >{} );
    std::sort( data + invert_at_pos, data + num_elements, std::greater< element_t >{} );
}

/**
 * Run the single-threaded CPU baseline that students are supposed to outperform
 * in order to obtain higher grades on this assignment. Times the execution and
 * prints to the standard output (e.g., the screen) that "wall time." Note that
 * the functions takes the input by value so as to not perturb the original data
 * in place.
 */
void run_cpu_baseline( std::vector< element_t > data, std::size_t switch_at, std::size_t n )
{
    auto const cpu_start = std::chrono::high_resolution_clock::now();
    opposing_sort( data.data(), switch_at, n );
    auto const cpu_end = std::chrono::high_resolution_clock::now();

    std::cout << "CPU Baseline time: "
              << std::chrono::duration_cast<std::chrono::nanoseconds>(cpu_end - cpu_start).count()
              << " ns" << std::endl;

    for( auto const x : data ) std::cout << x << " "; std::cout << std::endl;
}

} // namespace cpu
} // namespace a1
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "gpu_solution.cu"
/**
 * The file in which you will implement your GPU solutions!
 */

#include "algorithm_choices.h"

#include <chrono>    // for timing
#include <iostream>  // std::cout, std::endl

#include <algorithm>

#include "cuda_common.h"

namespace{
    int const MAX_WARPS = 32;
    int const MAX_LANES = 32;
}

namespace csc485b {
namespace a1      {
namespace gpu     {

/**
 * The CPU baseline benefits from warm caches because the data was generated on
 * the CPU. Run the data through the GPU once with some arbitrary logic to
 * ensure that the GPU cache is warm too and the comparison is more fair.
 */
__global__
void warm_the_gpu( element_t * data, std::size_t invert_at_pos, std::size_t num_elements )
{
    int const th_id = blockIdx.x * blockDim.x + threadIdx.x;

    // We know this will never be true, because of the data generator logic,
    // but I doubt that the compiler will figure it out. Thus every element
    // should be read, but none of them should be modified.
    if( th_id < num_elements && data[ th_id ] > num_elements * 100 )
    {
        ++data[ th_id ]; // should not be possible.
    }
}


/**
 * Your solution. Should match the CPU output.
 */
__global__
void opposing_sort( element_t * data, std::size_t invert_at_pos, std::size_t num_elements, int initial_stride, int sub_stride, int threads_per_block)
{
     __shared__ element_t smem [MAX_WARPS][MAX_LANES];

    int warp_id = threadIdx.y;
    int lane_id = threadIdx.x;
    int const th_id = (blockIdx.x * (blockDim.x * blockDim.y) + ((warp_id * blockDim.x) + lane_id));

    //if( th_id >= num_elements ){return;}

    // smem only contains data of threads and for threads of this block
    if ((th_id < num_elements) && (sub_stride < threads_per_block)){
        smem[warp_id][lane_id] = data[th_id];
    }

    __syncthreads();
    
    int global_group_size = initial_stride * 2;
    int group_number = th_id/global_group_size;
    int dir = (group_number % 2 == 0); //if 1 then ascending, 0 then descending

    int sub_group_size = sub_stride * 2;
    int head = (th_id / sub_group_size) * sub_group_size; //integer division

    if ((th_id < num_elements) && (th_id + sub_stride < num_elements) && 
        (th_id < head + (sub_group_size/2)) && (th_id + sub_stride < head + sub_group_size)) {

        // wrap around if (lane_id + stride) > 32
        int target_warp = warp_id;
        int target_lane = lane_id + sub_stride;
        target_warp += target_lane/32;
        target_lane = target_lane%32;
    
        // if the resulting warp_id is greater than the maximum warp allowed in the block (4), then read from global memory
        if (sub_stride >= threads_per_block) {

            int target_id = th_id + sub_stride;

            if (target_id < num_elements) {
                if ((data[th_id] < data[target_id]) != dir) {
                    element_t temp = data[th_id];
                    data[th_id] = data[target_id];
                    data[target_id] = temp;
                }
            }
        }
        else {
            if ((smem[warp_id][lane_id] < smem[target_warp][target_lane]) != dir) {
                element_t temp = smem[warp_id][lane_id];
                smem[warp_id][lane_id] = smem[target_warp][target_lane];
                smem[target_warp][target_lane] = temp;
                
            }
        }
    }

    __syncthreads();
    if ((th_id < num_elements) && (sub_stride < threads_per_block)){
        data[th_id] = smem[warp_id][lane_id];
    }
    
    return;
}

void final_step( element_t * data, std::size_t invert_at_pos, std::size_t num_elements){

}

/**
 * Performs all the logic of allocating device vectors and copying host/input
 * vectors to the device. Times the opposing_sort() kernel with wall time,
 * but excludes set up and tear down costs such as mallocs, frees, and memcpies.
 */
void run_gpu_soln( std::vector< element_t > data, std::size_t switch_at, std::size_t n )
{
    // Kernel launch configurations. Feel free to change these.
    // This is set to maximise the size of a thread block on a T4, but it hasn't
    // been tuned. It's not known if this is optimal.
    dim3 threads(MAX_LANES, MAX_WARPS);
    std::size_t const threads_per_block = threads.x * threads.y;
    std::size_t const num_blocks =  ( n + threads_per_block - 1 ) / threads_per_block;

    // Allocate arrays on the device/GPU
    element_t * d_data;
    cudaMalloc( (void**) & d_data, sizeof( element_t ) * n );
    CHECK_ERROR("Allocating input array on device");

    // Copy the input from the host to the device/GPU
    cudaMemcpy( d_data, data.data(), sizeof( element_t ) * n, cudaMemcpyHostToDevice );
    CHECK_ERROR("Copying input array to device");

    // Warm the cache on the GPU for a more fair comparison
    warm_the_gpu<<< num_blocks, threads>>>( d_data, switch_at, n );

    // Time the execution of the kernel that you implemented
    auto const kernel_start = std::chrono::high_resolution_clock::now();

    // start host loop
    for(int initial_stride = 1; initial_stride <= n/2; initial_stride <<= 1){
        for (int sub_stride = initial_stride; sub_stride>0; sub_stride >>= 1){
            opposing_sort<<< num_blocks, threads>>>( d_data, switch_at, n , initial_stride, sub_stride, threads_per_block);
            
        }
    }

    cudaDeviceSynchronize();

    // start host loop
    for(int initial_stride = 1; initial_stride <= n/2; initial_stride <<= 1){
        for (int sub_stride = initial_stride; sub_stride>0; sub_stride >>= 1){
            final_step<<<num_blocks, threads>>>( d_data, switch_at, n,  );
        }
    }

    cudaDeviceSynchronize();

    auto const kernel_end = std::chrono::high_resolution_clock::now();
    CHECK_ERROR("Executing kernel on device");

    // After the timer ends, copy the result back, free the device vector,
    // and echo out the timings and the results.
    cudaMemcpy( data.data(), d_data, sizeof( element_t ) * n, cudaMemcpyDeviceToHost );
    CHECK_ERROR("Transferring result back to host");
    cudaFree( d_data );
    CHECK_ERROR("Freeing device memory");

    std::cout << "GPU Solution time: "
              << std::chrono::duration_cast<std::chrono::nanoseconds>(kernel_end - kernel_start).count()
              << " ns" << std::endl;

    for( auto const x : data ) std::cout << x << " "; std::cout << std::endl;
}

} // namespace gpu
} // namespace a1
} // namespace csc485b

In [449]:
%%cuda_group_save -g "source" -n "main.cu"
/**
 * Driver for the benchmark comparison. Generates random data,
 * runs the CPU baseline, and then runs your code.
 */

#include <cstddef>  // std::size_t type
#include <iostream> // std::cout, std::endl
#include <vector>

#include "algorithm_choices.h"
#include "data_generator.h"
#include "data_types.h"
#include "cuda_common.h"

int main()
{
    // this part is 3 * n/4
    std::size_t const n = 512;
    std::size_t const switch_at = 3 * ( n >> 2 ) ;

    auto data = csc485b::a1::generate_uniform< element_t >( n );

    //print start data values
    for( auto const x : data ) std::cout << x << " "; std::cout << std::endl;

    csc485b::a1::cpu::run_cpu_baseline( data, switch_at, n );
    csc485b::a1::gpu::run_gpu_soln( data, switch_at, n );

    return EXIT_SUCCESS;
}

In [450]:
%cuda_group_run --group "source" --compiler-args "-O0 -g -std=c++20 -arch=sm_75"

20 364 206 623 823 834 944 3 170 816 175 106 502 558 580 214 210 889 544 6 311 133 26 436 524 118 163 304 432 158 903 240 432 340 668 536 41 513 993 365 148 464 244 152 232 486 288 532 610 61 613 664 142 24 371 573 381 728 235 101 382 808 594 526 792 356 727 499 749 645 496 759 977 308 753 999 466 614 415 808 461 885 826 1023 280 978 395 722 887 341 993 340 536 346 119 832 973 793 287 234 137 568 760 747 35 390 769 106 505 240 167 666 65 979 377 99 215 704 235 843 649 70 751 359 767 693 384 746 295 257 287 344 127 825 586 347 405 164 730 75 327 241 768 1013 55 712 689 157 50 278 408 962 35 664 242 471 733 662 624 555 650 138 371 642 384 187 111 828 119 818 671 840 786 863 756 469 422 441 297 221 705 1015 779 793 254 1002 478 775 707 349 89 625 761 418 1000 300 463 491 1020 897 955 997 884 891 734 805 610 495 904 741 32 768 192 106 33 34 395 697 630 52 923 221 279 672 310 566 916 501 685 742 116 603 807 818 279 620 9 503 408 19 740 442 58 513 498 256 538 855 428 358 631 690 99 448 670 1